In [67]:
%load_ext autoreload
%autoreload 2
import eval
import os
import glob
import pickle

from utils import protein
from utils.geometry import compute_rmsd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
motif = "3ixt"
motif_pdb = f"/data/cb/mihirb14/projects/BoltzDesign1/motifs/{motif}.pdb"
root_dir = f"./out/onemotif_twostates/{motif}/"

rows = []
for design_dir in sorted(glob.glob(os.path.join(root_dir, "design*"))):
    design_name = os.path.basename(design_dir)
    
    with open(os.path.join(design_dir, f"{motif}_spec.pkl"), "rb") as f:
        motif_mask = pickle.load(f)["motif_mask"]

    for state in [0, 1]:
        with open(os.path.join(design_dir, f"state{state}.pkl"),"rb") as f:
            outdict = pickle.load(f)
        for sample_idx in range(5): 
            pdb_file = os.path.join(design_dir, f"state{state}_sample{sample_idx}.pdb")
            if not os.path.exists(pdb_file):
                continue
            rmsd = eval.motifRMSD(motif_pdb, pdb_file, motif_mask)
            rows.append({
                "design": design_name,
                "state": state,
                "sample": sample_idx,
                "motifrmsd": rmsd.item() if hasattr(rmsd, "item") else float(rmsd),
                "plddt": outdict["plddt"].cpu().numpy()[sample_idx].mean(),
                "ptm": outdict["ptm"].cpu().numpy()[sample_idx]
                # prolly should save iptm iplddt for state 1
            })

df = pd.DataFrame(rows)
df.head()


,design,state,sample,motifrmsd,plddt,ptm
0,design0,0,0,0.824371,0.792900,0.604228
1,design0,0,1,0.888581,0.782107,0.590002
2,design0,0,2,0.853081,0.786949,0.626292
3,design0,0,3,0.945256,0.789702,0.601309
4,design0,0,4,0.810755,0.794108,0.626148


In [32]:
# design_dir = f"./out/onemotif_twostates/{motif}/design1/"

# with open(os.path.join(design_dir, f"state1.pkl"),"rb") as f:
#     out = pickle.load(f)
# out

In [91]:
# aggregate
agg = df.groupby(["design", "state"]).agg(
    motifRMSD_mean=("motifrmsd", "mean"),
    motifRMSD_std=("motifrmsd", "std"),
    plddt=("plddt", "mean"),
    ptm=("ptm", "mean")
).reset_index()

# pivot wider
pivot = agg.pivot(index="design", columns="state").reset_index()

# flatten multiindex -> "metric_state"
pivot.columns = ["_".join(map(str, col)).rstrip("_") for col in pivot.columns.to_flat_index()]

# clean up names
pivot = pivot.rename(columns=lambda c: c.replace("_0", "_unbound").replace("_1", "_bound"))
pivot

,design,motifRMSD_mean_unbound,motifRMSD_mean_bound,motifRMSD_std_unbound,motifRMSD_std_bound,plddt_unbound,plddt_bound,ptm_unbound,ptm_bound
0,design0,0.864409,1.522914,0.054154,0.053861,0.789153,0.665913,0.609596,0.647246
1,design1,0.463217,4.329279,0.028546,0.182347,0.774992,0.745027,0.562019,0.725640
2,design2,0.509397,1.382533,0.057338,0.163824,0.800471,0.580289,0.589939,0.454221
3,design3,2.413548,2.309627,0.154018,0.254213,0.592141,0.674145,0.333826,0.558182
4,design4,0.992195,2.377928,0.295492,0.570384,0.682384,0.569632,0.533953,0.398559
5,design5,0.764286,0.761084,0.098678,0.038225,0.677900,0.610815,0.618196,0.598148
6,design6,0.730642,3.213150,0.059604,0.102814,0.718641,0.689316,0.513290,0.653588
7,design7,6.468264,0.647563,0.409826,0.261952,0.646592,0.714662,0.388578,0.637311
8,design8,2.160156,4.319559,0.037607,0.241408,0.798116,0.583193,0.830599,0.525876
9,design9,1.454119,4.857787,1.530553,0.207612,0.589140,0.629815,0.301734,0.578672


In [92]:
import plotly.express as px

fig = px.scatter(
    pivot,
    x="motifRMSD_mean_unbound",
    y="motifRMSD_mean_bound",
    error_x="motifRMSD_std_unbound",
    error_y="motifRMSD_std_bound",
    color="plddt_unbound",
    hover_name="design",
    labels={
        "motifRMSD_mean_unbound": "Unbound motif RMSD",
        "motifRMSD_mean_bound": "Bound motif RMSD"
    },
    title="Unbound vs Bound motif RMSD (±1 std)",
    color_continuous_scale="sunsetdark"
)

fig.add_shape(type="line", x0=0, y0=0, x1=5, y1=5,
              line=dict(color="lightgray", dash="dash"))
fig.update_layout(
    xaxis=dict(range=[0,5]),
    yaxis=dict(range=[0,5], scaleanchor="x"),
    width=600, height=600,
    # plot_bgcolor="white",
    # paper_bgcolor="white"   
)
fig.update_traces(
    error_x=dict(color="gray", thickness=1.0),
    error_y=dict(color="gray", thickness=1.0)
)
fig.show()
